# 01. 미들웨어 시스템 심화 — v1 최대 신기능

## 학습 목표

- 에이전트 루프에서 미들웨어 훅의 역할과 실행 흐름을 이해한다
- 7가지 빌트인 미들웨어의 설정과 실전 사용법을 익힌다
- 데코레이터/클래스 기반 커스텀 미들웨어를 작성할 수 있다
- 다중 미들웨어 조합 시 실행 순서를 정확히 예측할 수 있다

## 1.1 환경 설정

미들웨어는 에이전트 실행의 각 단계에 훅(hook)을 삽입하여 모니터링, 변환, 신뢰성, 거버넌스를 구현하는 v1의 핵심 기능입니다. `create_agent` 함수의 `middleware` 파라미터에 미들웨어 인스턴스 리스트를 전달하여 사용합니다.

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

model = ChatOpenAI(model="gpt-4.1")

In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


## 1.2 미들웨어 아키텍처 개요

에이전트 루프는 **모델 호출 → 도구 선택 → 도구 실행 → 종료 판단**의 반복 사이클입니다. 미들웨어는 이 사이클의 각 단계에 훅(hook)을 삽입하여 세밀한 제어를 가능하게 합니다.

### 훅 유형

| 훅 유형 | 실행 시점 | 대표 용도 |
|---------|-----------|----------|
| `before_model` | 모델 호출 직전 | 프롬프트 수정, 로깅, 상태 업데이트 |
| `after_model` | 모델 응답 직후 | 응답 검증, 가드레일, 결과 변환 |
| `before_agent` | 에이전트 실행 시작 시 | 초기화, 전처리 |
| `after_agent` | 에이전트 실행 종료 시 | 정리, 후처리 |
| `wrap_model_call` | 모델 호출 감싸기 | 재시도, 캐싱, 폴백 |
| `wrap_tool_call` | 도구 호출 감싸기 | 도구 재시도, 감사 로그, 에러 핸들링 |

### 두 가지 훅 스타일

- **Node-style 훅** (`before_*`, `after_*`): 순차적으로 실행되며, 로깅/검증/상태 업데이트에 적합합니다.
- **Wrap-style 훅** (`wrap_*`): 핸들러(`next_fn`) 호출 여부를 제어할 수 있습니다. 0회(차단), 1회(통과), 다회(재시도) 호출이 가능하여 재시도, 캐싱, 변환 로직에 적합합니다.

미들웨어는 에이전트의 핵심 로직을 변경하지 않고도 모니터링, 변환, 신뢰성, 거버넌스 등 횡단 관심사(cross-cutting concerns)를 깔끔하게 분리할 수 있게 해줍니다.

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    SummarizationMiddleware,
    HumanInTheLoopMiddleware,
)

agent = create_agent(
    model="gpt-4.1", tools=[],
    middleware=[
        SummarizationMiddleware(model="gpt-4.1-mini", trigger=("messages", 50)),
        HumanInTheLoopMiddleware(interrupt_on={}),
    ],
)

## 1.3 SummarizationMiddleware

대화가 길어져 컨텍스트 윈도우를 초과할 때 자동으로 이전 대화를 요약하여 압축합니다. 장시간 실행되는 대화, 다중 턴 대화, 전체 대화 맥락 보존이 필요한 애플리케이션에 필수적입니다.

### 주요 파라미터

| 파라미터 | 설명 | 예시 |
|---------|------|------|
| `model` | 요약 생성에 사용할 경량 모델 (비용 절감) | `"gpt-4.1-mini"` |
| `trigger` | 요약 트리거 조건 | `("tokens", 4000)`, `("messages", 50)`, `("fraction", 0.8)` |
| `keep` | 요약 후 유지할 최근 컨텍스트 | `("messages", 20)` |
| `token_counter` | 커스텀 토큰 카운팅 함수 | 선택적 |
| `summary_prompt` | 커스텀 요약 프롬프트 템플릿 | 선택적 |

`trigger`는 토큰 수, 메시지 수, 윈도우 비율 중 하나를 기준으로 설정할 수 있으며, 조건에 도달하면 `keep`에 지정된 최근 메시지를 제외한 나머지를 요약문으로 교체합니다.

In [4]:
from langchain.agents.middleware import SummarizationMiddleware

summarizer = SummarizationMiddleware(
    model="gpt-4.1-mini",
    trigger=("tokens", 4000),
    keep=("messages", 20),
)

## 1.4 HumanInTheLoopMiddleware

고위험 도구 호출 전에 에이전트 실행을 중단하고 인간 승인을 기다립니다. 데이터베이스 쓰기, 금융 거래, 이메일 전송 등 고위험 작업이나 컴플라이언스 워크플로우에서 인간 감독이 필요한 경우에 사용합니다.

**`checkpointer` 필수** — 중단 후 상태를 복원하기 위해 체크포인터가 반드시 필요합니다.

### 결정 유형

| 결정 | 설명 | 사용 방법 |
|------|------|----------|
| `approve` | 도구 호출 승인 및 실행 | `Command(resume="approve")` |
| `edit` | 도구 인자 수정 후 실행 | `Command(resume={"type": "edit", "args": {...}})` |
| `reject` | 도구 호출 거부 | `Command(resume={"type": "reject", "reason": "..."})` |

`interrupt_on` 딕셔너리에서 각 도구별로 승인 정책을 설정합니다. `False`로 설정하면 해당 도구는 중단 없이 실행됩니다.

In [5]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

hitl = HumanInTheLoopMiddleware(
    interrupt_on={
        "send_email": {"allowed_decisions": ["approve", "edit", "reject"]},
        "read_email": False,
    }
)

In [6]:
agent = create_agent(
    model="gpt-4.1", tools=[],
    checkpointer=InMemorySaver(),
    middleware=[hitl],
)

In [7]:
from langgraph.types import Command

# result = agent.invoke(Command(resume="approve"), config=lf_config)
# result = agent.invoke(Command(resume={"type": "edit", "args": {"to": "new@ex.com"}}), config=lf_config)
# result = agent.invoke(Command(resume={"type": "reject", "reason": "필요 없음"}), config=lf_config)

## 1.5 ModelCallLimitMiddleware & ToolCallLimitMiddleware

무한 루프나 과도한 API 비용을 방지하기 위한 호출 제한 미들웨어입니다.

### ModelCallLimitMiddleware

에이전트가 모델을 호출하는 횟수를 제한합니다. 폭주하는 에이전트 방지, 프로덕션 비용 제어, 테스트 시 호출 예산 관리에 사용됩니다.

### ToolCallLimitMiddleware

도구 호출 횟수를 전역적으로 또는 특정 도구별로 제한합니다. 비용이 높은 외부 API 호출 제한, 검색/DB 쿼리 빈도 제어, 특정 도구의 레이트 리밋 적용에 유용합니다.

### 공통 파라미터

| 파라미터 | 설명 |
|---------|------|
| `thread_limit` | 전체 스레드(모든 invoke)에서의 최대 호출 수 |
| `run_limit` | 단일 invoke 실행에서의 최대 호출 수 |
| `exit_behavior` | `"end"` (정상 종료), `"error"` (예외 발생), `"continue"` (에러 메시지와 함께 계속 — ToolCallLimit 전용) |

ToolCallLimitMiddleware는 추가로 `tool_name` 파라미터를 받아 특정 도구에만 제한을 적용할 수 있습니다.

In [8]:
from langchain.agents.middleware import ModelCallLimitMiddleware

model_limit = ModelCallLimitMiddleware(
    thread_limit=10,
    run_limit=5,
    exit_behavior="end",
)

In [9]:
from langchain.agents.middleware import ToolCallLimitMiddleware

# 전역 제한
global_tool_limit = ToolCallLimitMiddleware(thread_limit=20, run_limit=10)

# 특정 도구 제한
search_limit = ToolCallLimitMiddleware(
    tool_name="search",
    thread_limit=5, run_limit=3,
    exit_behavior="continue",
)

## 1.6 ModelFallbackMiddleware

주 모델 실패 시 대체 모델 체인으로 자동 전환합니다. 프로덕션 장애 대응, 비용 최적화(비싼 모델 → 저렴한 모델 폴백), 멀티 프로바이더 중복성(OpenAI + Anthropic 등) 확보에 유용합니다.

생성자에 폴백 모델을 순서대로 전달하면, 주 모델 호출이 실패할 때 지정된 순서로 대체 모델을 시도합니다. 모든 폴백이 실패하면 최종 에러가 발생합니다.

In [10]:
from langchain.agents.middleware import ModelFallbackMiddleware

# gpt-4.1 실패 -> gpt-4.1-mini -> claude
fallback = ModelFallbackMiddleware(
    "gpt-4.1-mini",
    "claude-3-5-sonnet-20241022",
)

## 1.7 PIIMiddleware

개인 식별 정보(PII)를 자동 탐지하고 설정된 전략에 따라 처리합니다. 의료/금융 컴플라이언스, 고객 서비스 에이전트의 로그 세정, 민감한 사용자 데이터 처리 등에 필수적입니다.

### 빌트인 PII 타입
`email`, `credit_card`, `ip`, `mac_address`, `url`

### 처리 전략

| 전략 | 동작 | 예시 (이메일) |
|------|------|--------------|
| `block` | 예외 발생 — PII 발견 시 실행 중단 | 에러 발생 |
| `redact` | `[REDACTED_TYPE]`으로 교체 | `[REDACTED_EMAIL]` |
| `mask` | 부분 마스킹 | `u***@example.com` |
| `hash` | 결정적 해싱 | `a1b2c3d4...` |

### 적용 범위
- `apply_to_input`: 사용자 입력 메시지 검사
- `apply_to_output`: AI 응답 메시지 검사
- `apply_to_tool_results`: 도구 실행 결과 검사

### 커스텀 탐지기
빌트인 PII 타입 외에 세 가지 방식으로 커스텀 탐지기를 만들 수 있습니다:
1. **정규식 문자열**: 간단한 패턴 매칭
2. **컴파일된 정규식 (`re.compile`)**: 복잡한 정규식
3. **함수**: 검증 로직이 필요한 고급 탐지 (반환: `list[dict]` — `text`, `start`, `end` 키 포함)

In [11]:
from langchain.agents.middleware import PIIMiddleware

email_pii = PIIMiddleware("email", strategy="redact", apply_to_input=True)
card_pii = PIIMiddleware("credit_card", strategy="mask", apply_to_input=True)

In [12]:
# 커스텀 탐지기: 정규식 문자열
api_key_pii = PIIMiddleware(
    "api_key",
    detector=r"sk-[a-zA-Z0-9]{32}",
    strategy="block",
)

In [13]:
import re

# 커스텀 탐지기: 컴파일된 정규식
phone_pii = PIIMiddleware(
    "phone_number",
    detector=re.compile(r"\+?\d{1,3}[\s.-]?\d{3,4}[\s.-]?\d{4}"),
    strategy="mask",
)

In [14]:
# 커스텀 탐지기: 함수 (SSN 예시)
def detect_ssn(content: str) -> list[dict]:
    matches = []
    for m in re.finditer(r"\d{3}-\d{2}-\d{4}", content):
        first = int(m.group(0)[:3])
        if first not in [0, 666] and not (900 <= first <= 999):
            matches.append({"text": m.group(0), "start": m.start(), "end": m.end()})
    return matches

ssn_pii = PIIMiddleware("ssn", detector=detect_ssn, strategy="hash")

## 1.8 LLMToolSelectorMiddleware

도구가 10개 이상일 때, 경량 LLM이 사용자 쿼리를 분석하여 관련 도구만 선별합니다. 이를 통해 불필요한 도구 설명으로 인한 토큰 낭비를 줄이고, 모델이 관련 도구에 집중하여 정확도를 높일 수 있습니다.

### 주요 파라미터

| 파라미터 | 설명 | 기본값 |
|---------|------|--------|
| `model` | 도구 선택용 모델 | 에이전트의 메인 모델 |
| `system_prompt` | 커스텀 선택 지침 | 내장 프롬프트 |
| `max_tools` | 최대 선택 도구 수 | 전체 |
| `always_include` | 항상 포함할 도구 이름 리스트 | `[]` |

선택 모델로 `gpt-4.1-mini` 같은 경량 모델을 사용하면 비용을 절감하면서도 효과적인 도구 필터링이 가능합니다.

In [15]:
from langchain.agents.middleware import LLMToolSelectorMiddleware

tool_selector = LLMToolSelectorMiddleware(
    model="gpt-4.1-mini",
    max_tools=3,
    always_include=["search"],
)

## 1.9 커스텀 미들웨어 작성

두 가지 구현 방식이 있습니다:

### 1. 데코레이터 방식
단일 훅, 간단한 로직에 적합합니다. `@before_model`, `@after_model`, `@wrap_model_call`, `@wrap_tool_call` 데코레이터를 사용합니다.

### 2. 클래스 방식 (`AgentMiddleware`)
여러 훅을 조합하거나 설정이 필요한 경우 `AgentMiddleware`를 상속합니다. sync/async 구현을 동시에 제공할 수 있습니다.

### 커스텀 상태
미들웨어는 `NotRequired` 타입 힌트를 사용해 에이전트 상태를 확장할 수 있습니다. 이를 통해 실행 간 값 추적, 훅 간 데이터 공유, 레이트 리밋이나 감사 로깅 같은 횡단 관심사 구현이 가능합니다.

### 에이전트 점프
`after_model` 등에서 딕셔너리를 반환하여 에이전트 흐름을 제어할 수 있습니다:
- `{"jump_to": "end"}` — 에이전트 즉시 종료
- `{"jump_to": "tools"}` — 도구 실행 단계로 이동
- `{"jump_to": "model"}` — 모델 호출 단계로 이동

In [16]:
from langchain.agents.middleware import before_model

@before_model
def log_before(state, runtime):
    """모델 호출 전 메시지 수를 기록합니다."""
    print(f"[LOG] 메시지 {len(state.get('messages', []))}개")

In [17]:
from langchain.agents.middleware import after_model

@after_model
def validate_output(state, runtime):
    """가드: 금지된 콘텐츠를 차단합니다."""
    last = state["messages"][-1].content
    if "FORBIDDEN" in last:
        return {"jump_to": "end"}

In [18]:
from langchain.agents.middleware import wrap_model_call

@wrap_model_call
def retry_on_error(request, handler):
    """실패 시 모델 호출을 최대 2회 재시도합니다."""
    for attempt in range(3):
        try:
            return handler(request)
        except Exception as e:
            if attempt == 2: raise

In [19]:
from langchain.agents.middleware import AgentMiddleware

class AuditMiddleware(AgentMiddleware):
    def __init__(self, log_file="audit.log"):
        self.log_file = log_file
    def before_model(self, state, config):
        print(f"[AUDIT] before -> {self.log_file}")
    def after_model(self, state, config):
        print(f"[AUDIT] after -> {self.log_file}")

## 1.10 미들웨어 실행 순서

다중 미들웨어 등록 시 실행 순서를 정확히 이해해야 예상치 못한 동작을 방지할 수 있습니다.

`middleware=[A, B, C]` 등록 시:

![미들웨어 실행 순서 — before(순방향), after(역방향), wrap(중첩)](../assets/images/middleware_execution_order.png)

### 실전 팁
- **PII 검출은 로깅보다 먼저** 등록해야 로그에 PII가 포함되지 않습니다.
- **폴백 미들웨어는 재시도 미들웨어보다 뒤에** 배치하여, 재시도 실패 후 폴백이 작동하도록 합니다.
- `wrap` 훅에서 `next_fn`을 호출하지 않으면 이후 미들웨어와 실제 호출이 모두 건너뛰어집니다.

In [20]:
@before_model
def mw_a(state, runtime): print("before A")

@before_model
def mw_b(state, runtime): print("before B")

@before_model
def mw_c(state, runtime): print("before C")

# 실행: A -> B -> C (after_model이면 C -> B -> A)

## 1.11 미들웨어 조합 (Stacking)

프로덕션 환경에서는 여러 미들웨어를 함께 사용하여 종합적인 에이전트 거버넌스를 구현합니다. 미들웨어는 등록 순서에 따라 실행되므로, 보안(PII) → 신뢰성(폴백) → 비용 제어(호출 제한) → 컨텍스트 관리(요약) → 최적화(도구 선택) → 감독(HITL) 순으로 배치하는 것이 권장됩니다.

이러한 조합을 통해 각 미들웨어는 단일 책임 원칙을 유지하면서도, 전체적으로 강력한 프로덕션 에이전트 파이프라인을 구성할 수 있습니다.

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    PIIMiddleware, ModelFallbackMiddleware,
    ModelCallLimitMiddleware, SummarizationMiddleware,
    HumanInTheLoopMiddleware, LLMToolSelectorMiddleware,
)
from langgraph.checkpoint.memory import InMemorySaver

In [22]:
middleware_stack = [
    PIIMiddleware("email", strategy="redact", apply_to_input=True),
    ModelFallbackMiddleware("gpt-4.1-mini", "claude-3-5-sonnet-20241022"),
    ModelCallLimitMiddleware(thread_limit=50, run_limit=10),
    SummarizationMiddleware(model="gpt-4.1-mini", trigger=("tokens", 4000)),
]

production_agent = create_agent(
    model="gpt-4.1", tools=[], checkpointer=InMemorySaver(), middleware=middleware_stack,
)

## 요약

| 항목 | 핵심 내용 |
|------|----------|
| **아키텍처** | 4가지 훅: `before_model`, `after_model`, `wrap_model_call`, `wrap_tool_call` |
| **빌트인 7종** | Summarization, HITL, ModelCallLimit, ToolCallLimit, ModelFallback, PII, LLMToolSelector |
| **커스텀** | 데코레이터(`@before_model` 등) / `AgentMiddleware` 클래스 |
| **실행 순서** | `before`: 순방향, `after`: 역방향, `wrap`: 중첩 |
| **프로덕션** | PII → Fallback → Limit → Summarization → ToolSelector → HITL |

### 다음 단계
→ **[02_multi_agent_subagents.ipynb](./02_multi_agent_subagents.ipynb)**: 멀티에이전트: Subagents 패턴을 배웁니다.
